# 🔍 ANN Search Benchmark: TurboQuant vs FAISS
### Notebook 03 — GloVe-200 Recall & Speed · **GPU Recommended**

> **Run on Google Colab T4 or A100 for best results. CPU works but is slower.**

**What this notebook benchmarks:**
1. ✅ Download and prepare GloVe-200 (1M vectors, 200-dimensional)
2. ✅ Build FAISS exact index (ground truth upper bound)
3. ✅ Build FAISS-PQ index (baseline — requires codebook training)
4. ✅ Build TurboQuant index at 2-bit, 3-bit, 4-bit (zero training!)
5. ✅ Compute Recall@1, Recall@10, Recall@100
6. ✅ Speed benchmark (ms per query)
7. ✅ Memory comparison (bytes per vector)
8. ✅ Pareto frontier chart: Recall@10 vs Memory/vector

**Expected results (from TurboQuant paper):**
| Method | Bits | Recall@10 | Build Time | Memory/vec |
|--------|------|-----------|------------|------------|
| FAISS Exact | 32 | 1.00 | ~1s | 800 B |
| FAISS-PQ | 4 | 0.72–0.78 | **~120s** | 100 B |
| **TurboQuant** | **4** | **0.80–0.85** | **~2s** | ~110 B |

In [ ]:
# @title 🔍 Environment Check
import sys, torch
print(f'Python : {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
%%capture
# @title 📦 Install Dependencies
!pip install torch numpy scipy matplotlib seaborn tqdm
!pip install faiss-gpu  # use faiss-cpu if no GPU
import os
if not os.path.exists('/content/TQ-infer-engine'):
    !git clone https://github.com/Paramveersingh-S/TQ-infer-engine.git /content/TQ-infer-engine
!pip install -e /content/TQ-infer-engine -q

In [ ]:
# @title 📚 Imports
import sys, os, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

sys.path.insert(0, '/content/TQ-infer-engine')
from tqe.search.index import TurboQuantIndex
from tqe.search.query import benchmark_search_speed

try:
    import faiss
    HAS_FAISS = True
    print('✅ FAISS available')
except ImportError:
    HAS_FAISS = False
    print('⚠️  FAISS not available — skipping FAISS baselines')

sns.set_theme(style='whitegrid')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Benchmark config
DIM     = 200
N_TRAIN = 100_000  # use 1_000_000 on A100 for paper results
N_QUERY = 1_000
K_VALUES = [1, 10, 100]
print(f'Config: DIM={DIM}, N_TRAIN={N_TRAIN:,}, N_QUERY={N_QUERY:,}')

In [ ]:
# @title 📥 Section 1 — Download & Prepare GloVe-200
def load_glove_vectors(dim=200, n_total=110_000, cache_path='/content/glove_cache.npy'):
    """Load or generate GloVe-200 vectors."""
    if os.path.exists(cache_path):
        print(f'Loading cached vectors from {cache_path}')
        return np.load(cache_path)

    # Try official GloVe download first
    try:
        import urllib.request, zipfile
        url = 'https://nlp.stanford.edu/data/glove.6B.zip'
        print(f'Downloading GloVe from Stanford NLP...')
        urllib.request.urlretrieve(url, '/content/glove.6B.zip')
        with zipfile.ZipFile('/content/glove.6B.zip') as z:
            z.extract(f'glove.6B.{dim}d.txt', '/content')
        vecs = []
        with open(f'/content/glove.6B.{dim}d.txt') as f:
            for line in f:
                parts = line.strip().split()
                vecs.append([float(x) for x in parts[1:dim+1]])
        vecs = np.array(vecs[:n_total], dtype=np.float32)
    except Exception as e:
        print(f'GloVe download failed: {e}\nGenerating synthetic GloVe-like data...')
        np.random.seed(42)
        vecs = np.random.randn(n_total, dim).astype(np.float32)

    # Normalize to unit sphere (for inner product = cosine similarity)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True).clip(min=1e-8)
    vecs /= norms
    np.save(cache_path, vecs)
    print(f'✅ Vectors ready: shape={vecs.shape}')
    return vecs

all_vecs   = load_glove_vectors(DIM, N_TRAIN + N_QUERY)
train_vecs = all_vecs[:N_TRAIN].copy()
query_vecs = all_vecs[N_TRAIN : N_TRAIN + N_QUERY].copy()
train_t    = torch.from_numpy(train_vecs)
query_t    = torch.from_numpy(query_vecs).to(DEVICE)

print(f'Train: {train_vecs.shape}  |  Query: {query_vecs.shape}')
print(f'Memory: {train_vecs.nbytes / 1e6:.1f} MB (FP32 train set)')

In [ ]:
# @title 🎯 Section 2 — Build FAISS Exact Index (Ground Truth)
results_all = []
exact_index = None

if HAS_FAISS:
    t0 = time.perf_counter()
    exact_index = faiss.IndexFlatIP(DIM)
    exact_index.add(train_vecs)
    build_time = time.perf_counter() - t0
    print(f'✅ FAISS Exact Index built in {build_time:.2f}s')
    print(f'   Total vectors: {exact_index.ntotal:,}')

    # Compute ground truth for queries
    max_k = max(K_VALUES)
    _, gt_indices = exact_index.search(query_vecs, max_k)
    print(f'   Ground truth computed for {N_QUERY:,} queries')

    for k in K_VALUES:
        results_all.append({
            'method': 'FAISS Exact', 'bits': 32.0, 'k': k,
            'recall': 1.0, 'build_s': build_time,
            'mem_per_vec': DIM * 4, 'qps': None,
        })
else:
    # Brute-force ground truth with PyTorch
    print('Computing brute-force exact search...')
    exact_scores = torch.from_numpy(query_vecs) @ torch.from_numpy(train_vecs).T
    max_k = max(K_VALUES)
    _, gt_indices = exact_scores.topk(max_k, dim=-1)
    gt_indices = gt_indices.numpy()
    print('✅ Brute-force ground truth computed')

In [ ]:
# @title 📦 Section 3 — Build FAISS-PQ Index (Baseline)
if HAS_FAISS:
    print('Building FAISS-PQ index (4-bit equivalent)...')
    m_subq = max(1, DIM // 8)   # subquantizers
    t0 = time.perf_counter()
    pq_index = faiss.IndexPQ(DIM, m_subq, 8)  # m_subq × 8 bits = 4 bits/dim
    pq_index.train(train_vecs)
    pq_index.add(train_vecs)
    pq_build_s = time.perf_counter() - t0
    print(f'✅ FAISS-PQ built in {pq_build_s:.2f}s  (note: training took {pq_build_s:.1f}s)')

    def faiss_recall(index, gt_idx, queries, k):
        _, approx = index.search(queries, k)
        recalls = [len(set(gt_idx[i,:k].tolist()) & set(approx[i].tolist()))/k
                   for i in range(len(queries))]
        return float(np.mean(recalls))

    for k in K_VALUES:
        r = faiss_recall(pq_index, gt_indices, query_vecs, k)
        results_all.append({
            'method': 'FAISS-PQ', 'bits': 4.0, 'k': k,
            'recall': r, 'build_s': pq_build_s,
            'mem_per_vec': DIM // 2, 'qps': None,
        })
        print(f'  FAISS-PQ Recall@{k}: {r:.4f}')
else:
    print('FAISS not available — skipping FAISS-PQ')

In [ ]:
# @title ⚡ Section 4 — Build TurboQuant Indices (Multiple Bit-widths)
def compute_recall(tq_index, gt_indices, query_t, k):
    """Compute recall@k for TurboQuantIndex vs ground truth."""
    _, approx_idx = tq_index.search(query_t, k)
    approx_idx = approx_idx.cpu().numpy()
    recalls = [len(set(gt_indices[i,:k].tolist()) & set(approx_idx[i].tolist()))/k
               for i in range(len(query_t))]
    return float(np.mean(recalls))

for bits in [2.0, 3.0, 4.0]:
    print(f'\n--- TurboQuant {bits:.0f}-bit ---')
    idx = TurboQuantIndex(dim=DIM, bits_per_dim=bits, device=DEVICE)

    t0 = time.perf_counter()
    idx.add(train_t)
    build_s = time.perf_counter() - t0
    print(f'  Build time: {build_s:.2f}s  (no training needed!)')
    print(f'  Index memory: {idx.memory_bytes()/1e6:.2f} MB')
    mem_per_vec = idx.memory_bytes() // max(1, idx.ntotal())

    # Speed benchmark
    speed = benchmark_search_speed(idx, query_t[:100], k=10, n_repeats=20)
    qps   = speed['queries_per_second']
    print(f'  Speed: {qps:.0f} queries/sec ({speed["mean_latency_ms"]:.1f}ms mean)')

    for k in K_VALUES:
        r = compute_recall(idx, gt_indices, query_t, k)
        results_all.append({
            'method': f'TurboQuant', 'bits': bits, 'k': k,
            'recall': r, 'build_s': build_s,
            'mem_per_vec': mem_per_vec, 'qps': qps,
        })
        print(f'  Recall@{k}: {r:.4f}')

In [ ]:
# @title 📊 Section 5 — Recall@k Results Table
print('\n' + '='*72)
print(f'  ANN BENCHMARK RESULTS — GloVe-{DIM} ({N_TRAIN:,} vectors)')
print('='*72)
print(f'{'Method':18} {'Bits':>5} {'k':>4} {'Recall':>8} {'Build(s)':>10} {'Mem/vec':>9} {'QPS':>8}')
print('-'*72)
for r in results_all:
    qps_str = f"{r['qps']:.0f}" if r['qps'] else '—'
    print(f"{r['method']:18} {r['bits']:>5.1f} {r['k']:>4} {r['recall']:>8.4f} "
          f"{r['build_s']:>10.2f} {r['mem_per_vec']:>8}B {qps_str:>8}")
print('='*72)

In [ ]:
# @title 📈 Section 6 — Pareto Frontier: Recall@10 vs Memory per Vector
r10 = [r for r in results_all if r['k'] == 10]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: Pareto scatter ---
ax = axes[0]
colors_map = {'FAISS Exact': '#44BBA4', 'FAISS-PQ': '#F18F01', 'TurboQuant': '#2E86AB'}
marker_map  = {'FAISS Exact': '*',      'FAISS-PQ': 's',        'TurboQuant': 'o'}

plotted_labels = set()
for r in r10:
    label = f"{r['method']} {r['bits']:.0f}bit" if r['method'] != 'FAISS Exact' else r['method']
    color = colors_map.get(r['method'], '#999')
    mkr   = marker_map.get(r['method'], 'o')
    sc = ax.scatter(r['mem_per_vec'], r['recall'], s=200, color=color,
                    marker=mkr, zorder=5,
                    label=label if label not in plotted_labels else '_nolegend_')
    ax.annotate(label, (r['mem_per_vec'], r['recall']),
                textcoords='offset points', xytext=(5, 5), fontsize=8)
    plotted_labels.add(label)

ax.set_xlabel('Memory per Vector (bytes)', fontsize=11)
ax.set_ylabel('Recall@10', fontsize=11)
ax.set_title('Pareto Frontier: Recall@10 vs Memory', fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)

# --- Right: Recall@k bar comparison ---
ax2 = axes[1]
k_groups = sorted(set(r['k'] for r in results_all))
method_bits = [(r['method'], r['bits']) for r in results_all if r['k'] == k_groups[0]]

x = np.arange(len(k_groups))
bar_w = 0.8 / len(method_bits)
for i, (method, bits) in enumerate(method_bits):
    recalls_k = [r['recall'] for r in results_all if r['method'] == method and r['bits'] == bits]
    color = colors_map.get(method, '#999')
    alpha = 1.0 - (i * 0.15)
    label = f'{method} {bits:.0f}bit' if method != 'FAISS Exact' else method
    ax2.bar(x + i*bar_w - 0.4, recalls_k, bar_w, label=label,
            color=color, alpha=max(0.5, alpha), edgecolor='white')

ax2.set_xticks(x)
ax2.set_xticklabels([f'Recall@{k}' for k in k_groups])
ax2.set_ylabel('Recall')
ax2.set_title('Recall@k Comparison', fontsize=12)
ax2.set_ylim(0, 1.1)
ax2.legend(fontsize=8)

plt.suptitle(f'ANN Benchmark — GloVe-{DIM}, {N_TRAIN:,} train vectors', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('ann_benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# @title ⏱️ Section 7 — Build Time Comparison (Zero Preprocessing = Key Advantage)
build_data = {}
for r in results_all:
    key = f"{r['method']} {r['bits']:.0f}bit" if r['method'] != 'FAISS Exact' else r['method']
    build_data[key] = r['build_s']

fig, ax = plt.subplots(figsize=(9, 5))
labels = list(build_data.keys())
times  = list(build_data.values())
colors = ['#44BBA4' if 'Exact' in l else ('#F18F01' if 'PQ' in l) else '#2E86AB'
          for l in labels]
bars = ax.bar(labels, times, color=colors, edgecolor='white', linewidth=1.2)
for bar, t in zip(bars, times):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f'{t:.2f}s', ha='center', fontweight='bold', fontsize=10)
ax.set_ylabel('Build Time (seconds)')
ax.set_title('Index Build Time — TurboQuant needs NO training!')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('ann_build_time.png', dpi=150)
plt.show()

# Save results
import csv
os.makedirs('results', exist_ok=True)
with open('results/ann_benchmark.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=results_all[0].keys())
    writer.writeheader()
    writer.writerows(results_all)
print('📄 Saved: results/ann_benchmark.csv')
print('\n✅ ANN benchmark complete!')